# Análisis de errores

Notebook de propiedad de Pablo Linares (`docs/governance/04_EXECUTION_BRIEFS/PABLO_EXECUTION_BRIEF.md`), implementa la sección 6 del contrato de CLI y evaluación (`docs/governance/02_CONTRACTS/CLI_AND_EVALUATION_CONTRACT.md`):

> Seleccionar reproduciblemente 10–15 correctos y 10–15 incorrectos. Registrar `source_asset_id`, confianza, IoU, tipo de error e hipótesis. Categorías mínimas: omisión, falso positivo, fondo/iluminación, producto recortado, caja automática deficiente y dominio no representado.

**Estado actual: ejecuta sobre un fixture sintético, no sobre el dataset real.** BOX-001 (cajas) y SPL-001 (splits) todavía no producen datos reales, así que no hay `outputs/evaluation/baseline` real todavía. Este notebook:

1. Carga los artefactos que ya produce `src/evaluate.py` (congelados por el contrato): `metrics.json`, `error_examples.csv`, `predictions/predictions.json`.
2. Separa los ejemplos seleccionados en correctos / incorrectos (categorías automáticas: `correcto`, `omision`, `falso_positivo`).
3. Prepara la tabla de revisión manual para las cuatro categorías que exigen inspección visual humana (`fondo_iluminacion`, `producto_recortado`, `caja_automatica_deficiente`, `dominio_no_representado`) — ver `CATEGORIAS_MANUALES_NOTA` en `src/evaluate.py`.
4. Dibuja ground truth (verde) vs. predicción (rojo) sobre la imagen para cada ejemplo seleccionado, para apoyar esa revisión.

Todo el código de abajo ya corre, hoy, contra cualquier carpeta de salida que produzca `src/evaluate.py` (fixture o real) — solo cambia la celda de configuración de la sección siguiente. No usa ninguna dependencia nueva: `matplotlib`, `Pillow`, `PyYAML` y `numpy` ya están fijados por ENV-001 (`requirements.txt` / transitivas de `ultralytics`).

## 0. Configuración

Apuntar estas tres rutas a una corrida real de `src/evaluate.py` cuando exista (p. ej. `outputs/evaluation/baseline` sobre `configs/dataset.yaml`, split `test`, después de G4). Por defecto apuntan al fixture propio de TRN-001 para que el notebook sea ejecutable de punta a punta sin esperar a BOX-001/SPL-001.

Para generar el fixture y una evaluación de ejemplo localmente:

```bash
python tests/generate_fixture.py --salida tests/fixtures/mini_dataset --semilla 42
python src/train.py --config configs/baseline.yaml --data tests/fixtures/mini_dataset/dataset.yaml --output-dir outputs/runs/smoke --seed 42 --smoke
python src/evaluate.py --model outputs/runs/smoke/weights/best.pt --data tests/fixtures/mini_dataset/dataset.yaml --split val --output-dir outputs/evaluation/smoke
```

In [ ]:
import csv
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import yaml
from PIL import Image

# TODO-RESULTADOS: repuntar a la evaluacion real (outputs/evaluation/baseline,
# split test, --allow-test) una vez que BOX-001/SPL-001 esten disponibles y se
# haya pasado G4. Mientras tanto, el fixture propio de TRN-001 deja correr
# este notebook de punta a punta.
EVAL_DIR = Path("../outputs/evaluation/smoke")
DATASET_YAML = Path("../tests/fixtures/mini_dataset/dataset.yaml")
SPLIT = "val"

CATEGORIAS_MANUALES = [
    "fondo_iluminacion",
    "producto_recortado",
    "caja_automatica_deficiente",
    "dominio_no_representado",
]

## 1. Métricas resumen (`metrics.json`)

In [ ]:
with open(EVAL_DIR / "metrics.json") as f:
    metrics = json.load(f)

print(f"split={metrics['split']}  conf={metrics['conf_umbral']}  iou={metrics['iou_umbral']}")
for nombre, valor in metrics["metricas"].items():
    print(f"  {nombre:12s} = {valor:.4f}")
print(f"\n{metrics['nota_sku']}")

## 2. Ejemplos seleccionados (`error_examples.csv`)

Ya vienen seleccionados de forma determinística por `src/evaluate.py` (`seleccionar_ejemplos`): los correctos de mayor confianza, los incorrectos ordenados por `source_asset_id` — sin aleatoriedad ni selección manual en esta etapa.

In [ ]:
with open(EVAL_DIR / "error_examples.csv", newline="") as f:
    ejemplos = list(csv.DictReader(f))

with open(EVAL_DIR / "predictions" / "predictions.json") as f:
    predicciones = json.load(f)
pred_por_id = {p["source_asset_id"]: p for p in predicciones}

correctos = [e for e in ejemplos if e["categoria"] == "correcto"]
incorrectos = [e for e in ejemplos if e["categoria"] != "correcto"]

print(f"correctos: {len(correctos)}  incorrectos: {len(incorrectos)}  (total {len(ejemplos)})")

conteo_categorias = {}
for e in ejemplos:
    conteo_categorias[e["categoria"]] = conteo_categorias.get(e["categoria"], 0) + 1
print("por categoria automatica:", conteo_categorias)

## 3. Tabla de revisión manual

Las categorías automáticas (`correcto`, `omision`, `falso_positivo`) las calcula `src/evaluate.py` por IoU contra el ground truth — no requieren ojo humano. Las cuatro categorías manuales del contrato sí lo requieren y **no se infieren de este fixture sintético** (`CATEGORIAS_MANUALES_NOTA` en `src/evaluate.py`): un rectángulo de color sólido sobre fondo sólido no tiene ni iluminación real, ni recortes, ni dominio fuera de distribución que evaluar.

Esta celda arma la tabla que sí se completa a mano sobre datos reales: para cada ejemplo incorrecto, agrega columnas vacías `categoria_manual` (una de `CATEGORIAS_MANUALES`) y `notas_revision`, y la vuelca a CSV junto a los demás artefactos de la corrida para que quede junto a `error_examples.csv`, no perdida en el notebook.

In [ ]:
revision_manual = [
    {**e, "categoria_manual": "", "notas_revision": ""} for e in incorrectos
]

ruta_revision = EVAL_DIR / "error_examples_revision_manual.csv"
campos = ["source_asset_id", "categoria", "confianza", "iou", "hipotesis", "categoria_manual", "notas_revision"]
with open(ruta_revision, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=campos)
    writer.writeheader()
    for fila in revision_manual:
        writer.writerow(fila)

print(f"{len(revision_manual)} filas para revisión manual escritas en {ruta_revision}")
print(f"categorias_manual validas: {CATEGORIAS_MANUALES}")

## 4. Visualización: ground truth vs. predicción

Verde = ground truth (`labels/<split>/<source_asset_id>.txt`, formato YOLO). Rojo = predicción del modelo (`predictions.json`, coordenadas absolutas). Ayuda a decidir, a ojo, la `categoria_manual` de cada incorrecto.

In [ ]:
with open(DATASET_YAML) as f:
    ds = yaml.safe_load(f)
carpeta_base = Path(ds.get("path", DATASET_YAML.parent))
carpeta_imagenes = carpeta_base / ds[SPLIT]
# misma convencion de resolucion de labels que src/evaluate.py (post EVAL-001):
# labels vive al lado de images, mismo split.
carpeta_labels = Path(str(carpeta_imagenes).replace(
    os.sep + "images" + os.sep, os.sep + "labels" + os.sep
))


def leer_gt_yolo(ruta_label):
    cajas = []
    if not ruta_label.is_file():
        return cajas
    with open(ruta_label) as f:
        for linea in f:
            partes = linea.strip().split()
            if len(partes) != 5:
                continue
            _, cx, cy, w, h = partes
            cajas.append((float(cx), float(cy), float(w), float(h)))
    return cajas


def yolo_a_xyxy(caja, img_w, img_h):
    cx, cy, w, h = caja
    xmin = (cx - w / 2) * img_w
    ymin = (cy - h / 2) * img_h
    xmax = (cx + w / 2) * img_w
    ymax = (cy + h / 2) * img_h
    return xmin, ymin, xmax, ymax


def dibujar_comparacion(ejemplo, ax):
    sid = ejemplo["source_asset_id"]
    pred = pred_por_id[sid]
    img = Image.open(carpeta_imagenes / pred["imagen"]).convert("RGB")
    ax.imshow(img)
    w, h = img.size

    for caja in leer_gt_yolo(carpeta_labels / f"{sid}.txt"):
        xmin, ymin, xmax, ymax = yolo_a_xyxy(caja, w, h)
        ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                    fill=False, edgecolor="lime", linewidth=2))
    for caja_pred in pred["predicciones"]:
        xmin, ymin, xmax, ymax = caja_pred["xyxy"]
        ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin,
                                    fill=False, edgecolor="red", linewidth=2))

    conf = ejemplo["confianza"] or "-"
    ax.set_title(f"{sid}\n{ejemplo['categoria']} (conf={conf}, iou={ejemplo['iou']})", fontsize=8)
    ax.axis("off")


muestras = (correctos + incorrectos)[:6]
if muestras:
    fig, axes = plt.subplots(1, len(muestras), figsize=(3 * len(muestras), 3))
    axes = [axes] if len(muestras) == 1 else axes
    for ax, ejemplo in zip(axes, muestras):
        dibujar_comparacion(ejemplo, ax)
    fig.tight_layout()
else:
    print("no hay ejemplos seleccionados para visualizar")

## 5. Siguientes pasos

- Repuntar `EVAL_DIR`/`DATASET_YAML`/`SPLIT` a la evaluación real (`outputs/evaluation/baseline`, split `test`) cuando BOX-001/SPL-001 entreguen datos reales y se pase G4.
- Completar a mano `categoria_manual` y `notas_revision` en `error_examples_revision_manual.csv` usando las imágenes de la sección 4.
- Volcar el resumen (conteos por categoría, 2-3 ejemplos representativos) a la sección de Conclusiones/Resultados de `informe/main.tex` (marcador `% TODO-RESULTADOS`).